In [1]:
%load_ext tensorboard
!rm -rf "./logs"

import os
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from datetime import datetime
from tensorflow import keras
from tensorflow.keras import layers
from tensorboard.plugins.hparams import api as hp

import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append("../")

from aux_func import *
from models import *

print(tf.version.VERSION)
print(keras.__version__)

gpus = tf.config.list_physical_devices("GPU")        
log_gpus = tf.config.list_logical_devices('GPU')
print(len(gpus), "Physical GPUs,", len(log_gpus), "Logical GPU")

from psutil import *

!lscpu|grep "Model name"
!nvidia-smi -L

2.7.0
2.7.0
1 Physical GPUs, 1 Logical GPU
Model name:                      Intel(R) Core(TM) i7-1065G7 CPU @ 1.30GHz
GPU 0: NVIDIA GeForce MX330 (UUID: GPU-4d2220a0-a8ad-9e2b-c6d3-6c49af0eb31b)


2021-12-27 13:27:50.401843: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:939] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2021-12-27 13:27:50.427819: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:939] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2021-12-27 13:27:50.428026: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:939] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2021-12-27 13:27:50.429526: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compil

In [6]:
# data = pd.read_csv(
#     "../time-basis-app/time_train_data.csv",
#      sep=";")

# tdata = pd.read_csv(
#     "../time-basis-app/time_test_data.csv",
#      sep=";")

# hdata = pd.read_csv(
#     "../time-basis-app/time_hold_data.csv",
#      sep=";")


data = pd.read_csv(
    "../last-event-app/random_train_data.csv",
     sep=";")

tdata = pd.read_csv(
    "../last-event-app/random_hold_data.csv",
     sep=";")

In [7]:
df = data.copy()
t_df = tdata.copy()
#h_df = hdata.copy()

In [14]:
def model_training(model, train_x, train_y, val_split, batch_size):
  
    es = keras.callbacks.EarlyStopping(monitor="val_loss",
                                       mode="min",
                                       verbose=1,
                                       patience=3)
    
    hist = model.fit(train_x, train_y,
                     batch_size=batch_size,
                     epochs=200,
                     verbose=0,
                     validation_split=val_split,
                     callbacks=[es])
    return hist

def model_eval_timebasis(model, test_x, test_y, k=5):
    hit = 0
    cus = 0
    mrr = 0
    retk = set()
    relk = set()
    p = 0
    r = 0
    y_pred = model.predict(test_x)
    cust = test_data["c_id"].unique()
    counter = len(cust)
    for i in range(counter):
        c = cust[i]
        rel = hold_data.loc[hold_data["c_id"]==c, "item_code"]
        if len(rel) > 0:
            ret = y_pred[i][-1].argsort()[::-1][:k]
            cus += 1
            retk = retk.union(set(ret))
            inter = set.intersection(rel, ret)
            if len(inter) > 0:
                rank = 5
                for j in inter:
                    th = np.where(ret==j)
                    if th[0][0]<=rank:
                        rank = th[0][0]  
                mrr += 1/(rank+1)
                hit += 1
                p += len(inter)/len(ret)
                r += len(inter)/len(rel)

    return p/cus, r/cus, mrr/cus, hit/cus, len(retk)

def model_eval_lastevent(model, test_x, test_y, k=5):
    hit = 0
    cus = 0
    mrr = 0
    retk = set()
    relk = set()
    y_pred = model.predict(test_x)
    cust = test_data["c_id"].unique()
    counter = len(cust)
    for i in range(counter):
        c = cust[i]
        rel = test_y[i][-1]
        ret = y_pred[i][-1].argsort()[::-1][:k]
        retk = retk.union(set(ret))
        r = np.where(ret==int(rel))
        if len(r[0]) > 0:
            rr = 1/(r[0][0]+1)
            hit += 1
        else:
            rr = 0
        mrr = mrr + rr

    return hit/counter, mrr/counter, len(retk)

def get_inputs(hparams):
    if hparams["model"]==1:
        input_list=[hparams["train_list"][0]]
        eval_list=[hparams["test_list"][0]]
    elif hparams["model"]==2:
        input_list=[hparams["train_list"][0], hparams["train_list"][1]]
        eval_list=[hparams["test_list"][0], hparams["test_list"][1]]
    elif hparams["model"]==3:
        input_list=[hparams["train_list"][0], hparams["train_list"][1],
                    hparams["train_list"][5]]
        eval_list=[hparams["test_list"][0], hparams["test_list"][1],
                   hparams["test_list"][5]]
    elif hparams["model"]==4:
        input_list=[hparams["train_list"][0], hparams["train_list"][1],
                    hparams["train_list"][5], hparams["train_list"][6]]
        eval_list=[hparams["test_list"][0], hparams["test_list"][1],
                   hparams["test_list"][5], hparams["test_list"][6]]
    elif hparams["model"]==5:
        input_list=[hparams["train_list"][0], hparams["train_list"][1],
                    hparams["train_list"][5], hparams["train_list"][6],
                    hparams["train_list"][2]]
        eval_list=[hparams["test_list"][0], hparams["test_list"][1],
                   hparams["test_list"][5], hparams["test_list"][6],
                   hparams["test_list"][2]]
    elif hparams["model"]==6:
        input_list=[hparams["train_list"][0], hparams["train_list"][1],
                    hparams["train_list"][5], hparams["train_list"][6],
                    hparams["train_list"][4]]
        eval_list=[hparams["test_list"][0], hparams["test_list"][1],
                   hparams["test_list"][5], hparams["test_list"][6],
                   hparams["test_list"][4]]
    elif hparams["model"]==7:
        input_list=[hparams["train_list"][0], hparams["train_list"][1],
                    hparams["train_list"][5], hparams["train_list"][6],
                    hparams["train_list"][3]]
        eval_list=[hparams["test_list"][0], hparams["test_list"][1],
                   hparams["test_list"][5], hparams["test_list"][6],
                   hparams["test_list"][3]]
    else:
        input_list=[hparams["train_list"][0], hparams["train_list"][1],
                    hparams["train_list"][5], hparams["train_list"][6],
                    hparams["train_list"][2], hparams["train_list"][3]]
        eval_list=[hparams["test_list"][0], hparams["test_list"][1],
                   hparams["test_list"][5], hparams["test_list"][6],
                   hparams["test_list"][2], hparams["test_list"][3]]

    return input_list, eval_list

def run(hparams, model, time=False):
    print(f"Experiment:{hparams['model']}-{hparams['style']}-{hparams['n_padded']}-{hparams['k']}-{hparams['replication']}")
    input_list, eval_list = get_inputs(hparams)
    batch_size = 128
    val_split = 0.2
    if time:
        val_split = 0.1
    hist = model_training(model,
                          input_list,
                          hparams["train_list"][-1],
                          val_split,
                          batch_size)
    if time:
        pre, rec, mrr, hit, retk = model_eval_timebasis(hist.model,
                                                        eval_list,
                                                        hparams["test_list"][-1],
                                                        hparams["k"])
        return pre, rec, mrr, hit, retk
    else:
        hit, mrr,retk = model_eval_lastevent(hist.model,
                                        eval_list,
                                        hparams["test_list"][-1],
                                        hparams["k"])
        return hit, mrr, retk

In [15]:
df_a = df.append(t_df, ignore_index=True)
pro_max = df_a["item_code"].astype("int64").max()
rec_max = df_a["recency"].astype("int64").max()
pay_max = df_a["payment"].astype("int64").max()
cat_max = df_a["category_code"].astype("int64").max()
met_max = df_a["method_code"].astype("int64").max()
mon_max = df_a["month"].astype("int64").max()
day_max = df_a["dayofweek"].astype("int64").max()

hold_c = df_a["c_id"].unique()

emb_units = [96, 64, 128, 96, 64, 128, 128, 64]
rnn_units = [100, 100, 200, 200, 200, 200, 200, 200]
drop_rates = [0.8, 0.6, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8]
n_paddeds = [24, 32]
styles = ["lstm"]

exp_df = pd.DataFrame(columns=["model", "n_padded", "k", "style",
                               "replication", "mrr", "hit", "coverage"])

for n_padded in n_paddeds:
    for n in range(1,9):
        for s in styles:
            for r in range(1,11):
                train_c, test_c = train_test_split(hold_c, test_size=0.2, shuffle=True)
                train_df = df_a[df_a["c_id"].isin(train_c)]
                test_df = df_a[df_a["c_id"].isin(test_c)]
                train_data = create_sequences(train_df)
                train_data = create_splits(train_data)
                train_list = make_padding(train_data, n_padded)
                test_data = create_sequences(test_df)
                test_data = create_splits(test_data)
                test_list = make_padding(test_data, n_padded)
                hparams = {
                    "bidirect": False,
                    "style": s,
                    "model": n,
                    "emb_unit": emb_units[n],
                    "rnn_unit": rnn_units[n],
                    "dropout": drop_rates[n],
                    "learning": 0.001,
                    "replication": r,
                    "n_padded": n_padded,
                    "item_max": pro_max+1,
                    "rec_max": rec_max+1,
                    "pay_max": pay_max+1,
                    "cat_max": cat_max+1,
                    "met_max": met_max+1,
                    "mon_max": mon_max+1,
                    "day_max": day_max+1,
                    "k": 5,
                    "train_list": train_list,
                    "test_list": test_list
                }
                model = model_base(hparams)
                hit, mrr, retk = run(hparams, model)
                exp_df = exp_df.append(
                                    {"model": n,
                                    "n_padded": n_padded,
                                    "k": 5,
                                    "style": s,
                                    "replication":r,
                                    "mrr": mrr,
                                    "hit": hit,
                                    "coverage": retk/pro_max}, ignore_index=True)


exp_df.to_csv("experiment_results.csv", sep=";", index=False)
exp_df

Experiment:1-lstm-24-5-1


2021-12-27 15:04:30.693877: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:689] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "NVIDIA GeForce MX330" frequency: 1594 num_cores: 3 environment { key: "architecture" value: "6.1" } environment { key: "cuda" value: "11020" } environment { key: "cudnn" value: "8100" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 524288 shared_memory_size_per_multiprocessor: 98304 memory_size: 1288044544 bandwidth: 56064000 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }
2021-12-27 15:04:31.735393: I tensorflow/stream_executor/cuda/cuda_dnn.cc:366] Loaded cuDNN version 8301
2021-12-27 15:04:42.969312: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:689] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT 

Epoch 00013: early stopping


2021-12-27 15:07:14.202573: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:689] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "NVIDIA GeForce MX330" frequency: 1594 num_cores: 3 environment { key: "architecture" value: "6.1" } environment { key: "cuda" value: "11020" } environment { key: "cudnn" value: "8100" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 524288 shared_memory_size_per_multiprocessor: 98304 memory_size: 1288044544 bandwidth: 56064000 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }
2021-12-27 15:07:24.599034: W tensorflow/core/common_runtime/bfc_allocator.cc:462] Allocator (GPU_0_bfc) ran out of memory trying to allocate 9.50MiB (rounded to 9956608)requested by op CudnnRNN
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve th

InternalError:    Failed to call ThenRnnForward with model config: [rnn_mode, rnn_input_mode, rnn_direction_mode]: 2, 0, 0 , [num_layers, input_size, num_units, dir_count, max_seq_length, batch_size, cell_num_units]: [1, 64, 100, 1, 24, 32, 100] 
	 [[{{node CudnnRNN}}]]
	 [[model_1/lstm_1/PartitionedCall]] [Op:__inference_predict_function_11691]

Function call stack:
predict_function -> predict_function -> predict_function
